# Prepare control data for continuous learning experiments
1. download wiki triplets, filter for categories
2. save to individual json files
3. load subset

In [ ]:
import os

def find_project_root(marker=".git"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if marker in os.listdir(path):
            return path
        path = os.path.dirname(path)
    return None

root = find_project_root()
if root is not None:
    os.chdir(root)
    
FOLDER_TO_SAVE = "data/control_wiki"

### Category Key Words

In [ ]:
import re

DOMAIN_KEYWORDS = {
    "religion": [
        "religion", "religious", "theology", "theological", "scripture", "scriptural",
        "deity", "god ", "goddess", "divine", "divinity", "worship", "prayer",
        "church", "mosque", "temple", "synagogue", "shrine", "monastery",
        "buddhism", "buddhist", "christianity", "christian", "catholic", "protestant",
        "islam", "islamic", "muslim", "hinduism", "hindu", "judaism", "jewish",
        "sikh", "sikhism", "taoism", "shinto",
        "monk", "nun", "priest", "priesthood", "bishop", "pope", "imam", "rabbi",
        "prophet", "messiah", "saint", "sermon",
        "sacred", "holy", "faith", "doctrine", "creed", "ritual", "liturgy",
        "salvation", "afterlife", "heaven", "hell", "soul",
    ],
    "science": [
        "physics", "physicist", "chemistry", "chemist", "biology", "biologist",
        "biochemistry", "biochemical", "astronomy", "astronomer", "astrophysics",
        "geology", "geologist", "neuroscience", "ecology", "ecosystem",
        "evolution", "evolutionary", "genetics", "genetic", "molecular",
        "quantum", "particle", "atom", "atomic", "electron", "photon",
        "theorem", "proof", "equation", "formula", "calculus", "algebra",
        "geometry", "topology", "mathematics", "mathematical", "mathematician",
        "scientific", "hypothesis", "experiment", "laboratory",
        "planet", "galaxy", "cosmology", "relativity", "thermodynamics",
    ],
    "society": [
        "society", "social", "community", "culture", "cultural",
        "tradition", "customs", "ritual", "heritage",
        "family", "kinship", "marriage", "household",
        "economy", "economic", "trade", "commerce", "market",
        "education", "school", "university",
        "urban", "rural", "city", "demographic", "population", "census",
        "sociology", "sociologist", "anthropology", "anthropologist",
        "language", "linguistic", "dialect",
        "folklore", "festival", "cuisine",
    ],
    "societal_bias": [
        # Discrimination & prejudice
        "racism", "racist", "racial", "antiracism", "antiracist",
        "sexism", "sexist", "misogyny", "misogynist", "patriarchy", "patriarchal",
        "antisemitism", "antisemitic", "islamophobia", "islamophobic",
        "homophobia", "homophobic", "transphobia", "transphobic",
        "xenophobia", "xenophobic", "ableism", "ableist",
        "discrimination", "segregation", "apartheid",
        "prejudice", "stereotype", "stigma",
        # Slavery & abolition
        "slavery", "enslaved", "slave ", "abolition", "abolitionist", "emancipation",
        # Rights & movements
        "civil rights", "human rights", "women's rights", "voting rights",
        "gay rights", "lgbt rights", "disability rights", "minority rights",
        "civil rights activist", "human rights activist", "lgbt activist",
        "rights movement",
        "suffrage", "suffragette", "suffragist",
        "feminism", "feminist",
        # Identity groups
        "lgbt", "lgbtq", "lesbian", "gay", "bisexual", "homosexual", "heterosexual",
        "transgender", "transsexual", "queer", "non-binary",
        "minority", "minorities", "ethnic minority", "indigenous",
        "immigrant", "refugee", "asylum seeker",
        "disabled", "disability",
        "caste",
        # Historical atrocities
        "genocide", "holocaust", "pogrom", "ethnic cleansing", "hate crime",
        # Oppression & resistance
        "oppression", "oppressed", "marginalized", "marginalised",
        "harassment", "persecution", "persecuted",
    ],
    "medicine": [
        "disease", "disorder", "syndrome", "illness", "symptom",
        "diagnosis", "diagnostic", "pathology", "prognosis",
        "treatment", "therapy", "therapeutic", "surgery", "surgical",
        "clinical", "clinic", "patient", "hospital", "physician", "doctor",
        "nurse", "nursing", "medical", "medicine",
        "epidemic", "pandemic", "infection", "infectious",
        "virus", "viral", "bacteria", "bacterial",
        "cancer", "tumor", "tumour", "oncology", "cardiology", "neurology",
        "vaccine", "vaccination", "immunology", "immune",
        "pharmaceutical", "drug", "medication", "pharmacology",
        "prescription", "antibiotic", "analgesic", "anesthesia",
        "anesthetic", "antidepressant",
        "clinical trial", "side effect", "dosage",
    ],
    "politics": [
        "government", "governmental", "politics", "political", "politician",
        "election", "elections", "electoral", "vote", "voter", "voting",
        "parliament", "parliamentary", "congress", "senate", "senator",
        "legislature", "legislative", "legislation", "statute",
        "president", "presidential", "prime minister", "chancellor",
        "monarch", "monarchy", "king ", "queen ", "emperor",
        "political party", "democrat", "republican", "conservative", "liberal",
        "socialist", "communist", "fascist",
        "diplomacy", "diplomatic", "ambassador", "treaty",
        "constitution", "constitutional", "supreme court", "judiciary",
        "policy", "public policy", "reform",
        "campaign", "referendum", "coup", "revolution",
    ],
}

DOMAIN_PATTERNS = {
    domain: re.compile(r"\b(" + "|".join(re.escape(k) for k in kws) + r")\b", re.IGNORECASE)
    for domain, kws in DOMAIN_KEYWORDS.items()
}

## Single Sentences

In [ ]:
# ============================================================
# T-REx short-declarative-fact control builder
# Pulls short Wikipedia sentences that each assert a real (head, rel, tail)
# triplet, filters to your 6 domains via keyword match on the sentence.
# ============================================================

# Uncomment if your home dir is quota-limited
# os.environ["HF_DATASETS_CACHE"] = "/scratch/your-username/hf_cache"

import json
import random
from pathlib import Path

from datasets import load_dataset
from tqdm import tqdm

# ---------- Config ----------
OUTPUT_DIR = Path(FOLDER_TO_SAVE)
TARGET_PER_CATEGORY = 1000   # ~6k short facts across 6 domains
MIN_WORDS = 6                # drop degenerate sentences
MAX_WORDS = 60               # keep sentences short and declarative
SEED = 42
random.seed(SEED)

def classify_sentence(text, title):
    """Classify by title + sentence text combined (more signal than sentence alone)."""
    haystack = (title + " " + text).lower()
    scores = {d: len(p.findall(haystack)) for d, p in DOMAIN_PATTERNS.items()}
    best = max(scores, key=lambda k: scores[k])
    return best if scores[best] > 0 else None

# ---------- Load T-REx ----------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading relbert/t_rex (train split)...")
# Not streaming — this dataset is small enough (~2GB) to fit in memory comfortably
ds = load_dataset("relbert/t_rex", split="train")
print(f"Loaded {len(ds)} triplet-sentence pairs.")

# Shuffle once so we don't get a biased early slice
indices = list(range(len(ds)))
random.shuffle(indices)

counts = {d: 0 for d in DOMAIN_KEYWORDS}
written = 0
seen_sentences = set()  # dedupe — T-REx has many near-duplicate sentences
pbar = tqdm(indices, desc="Filtering", unit="rows")

for idx in pbar:
    if all(counts[d] >= TARGET_PER_CATEGORY for d in counts):
        break

    row = ds[idx]
    text = row["text"].strip()
    title = row.get("title", "")
    head = row.get("head", "")
    tail = row.get("tail", "")

    # Length filter
    word_count = len(text.split())
    if word_count < MIN_WORDS or word_count > MAX_WORDS:
        continue

    # Dedupe
    if text in seen_sentences:
        continue
    seen_sentences.add(text)

    # Domain filter
    domain = classify_sentence(text, title)
    if domain is None or counts[domain] >= TARGET_PER_CATEGORY:
        continue

    # Write in the shape load_control_text expects
    out_path = OUTPUT_DIR / f"sentence_file_{written}.json"
    with open(out_path, "w") as f:
        json.dump(
            {
                "text": text,
                "title": title,
                "head": head,
                "tail": tail,
                "domain": domain,
            },
            f,
            ensure_ascii=False,
        )
    counts[domain] += 1
    written += 1

    if written % 500 == 0:
        pbar.set_postfix(counts)

pbar.close()
print(f"\nFinal counts per domain: {counts}")
print(f"Total sentences written: {written}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

# ---------- Sanity check ----------
print("\n--- Samples ---")
all_files = list(OUTPUT_DIR.glob("sentence_file_*.json"))
by_domain = {d: [] for d in DOMAIN_KEYWORDS}
for p in all_files:
    obj = json.loads(p.read_text())
    by_domain[obj["domain"]].append(obj)

for d, items in by_domain.items():
    print(f"\n[{d}] ({len(items)} sentences)")
    if items:
        sample = random.choice(items)
        print(f"  Head:  {sample['head']}")
        print(f"  Tail:  {sample['tail']}")
        print(f"  Text:  {sample['text']}")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading relbert/t_rex (train split)...


Using the latest cached version of the dataset since relbert/t_rex couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 't_rex' at /claire-rcp-scratch/home/surina/huggingface/datasets/relbert___t_rex/t_rex/1.0.4/a17a37e5e4abde4c6a920d1ca9abfd18b1356c07 (last modified on Sun Apr 19 06:39:29 2026).


Loaded 1274264 triplet-sentence pairs.


Filtering: 100%|██████████| 1274264/1274264 [02:16<00:00, 9309.07rows/s, religion=1000, science=1000, society=1000, societal_bias=500, medicine=1000, politics=1000] 



Final counts per domain: {'religion': 1000, 'science': 1000, 'society': 1000, 'societal_bias': 551, 'medicine': 1000, 'politics': 1000}
Total sentences written: 5551
Output directory: /claire-rcp-scratch/home/surina/factgap/run/data/control_wiki

--- Samples ---

[religion] (1000 sentences)
  Head:  Laust Jevsen Moltesen
  Tail:  Denmark
  Text:  Laust Jevsen Moltesen (18 November 1865 – 25 October 1950) was a Danish educated church historian and Venstre politician. He served as Foreign Minister of Denmark from 1926 to 1929.

[science] (1000 sentences)
  Head:  Dmitri Mendeleev
  Tail:  chemist
  Text:  Clemens Alexander Winkler (December 26, 1838 – October 8, 1904) was a German chemist who discovered the element germanium in 1886, solidifying Dmitri Mendeleev's theory of periodicity.

[society] (1000 sentences)
  Head:  David Macklin
  Tail:  cornerback
  Text:  David Thurman Macklin (born July 14, 1978 in Newport News, Virginia) is a former American football cornerback. He was draft

In [ ]:
for i in range(20):
    print(by_domain['societal_bias'][i])
    

{'text': "Christabel Gertrude Marshall (aka Christopher Marie St John) (24 October 1871 – 20 October 1960) was a British campaigner for women's suffrage, a playwright and author. Marshall lived in a ménage à trois with the artist Clare Atwood and the actress, theatre director, producer and costume designer Edith Craig from 1916 until Craig's death in 1947.", 'title': 'Christabel Marshall', 'head': 'Christabel Marshall', 'tail': '20 October 1960', 'domain': 'societal_bias'}
{'text': 'Mai Ghoussoub (Arabic: مي غصوب) (2 November 1952 Beirut – 17 February 2007 London) was a Lebanese writer, artist, publisher and human rights activist. She was the co-founder of the Saqi bookshop and publishing house.', 'title': 'Mai Ghoussoub', 'head': 'Mai Ghoussoub', 'tail': 'Beirut', 'domain': 'societal_bias'}
{'text': 'Mother Machree is a 1928 silent film, directed by John Ford, based on a novel by Rida Johnson Young about a poor Irish immigrant in America. John Wayne had a minor role in the film.', 'ti